# Circuit Motif Discovery — Full Colab Pipeline

Run all steps from environment setup to final figures.

In [ ]:
# Cell 1: Setup
!nvidia-smi

# Option A: clone repo
!git clone https://github.com/YOUR_USERNAME/circuit-motif-discovery.git
%cd circuit-motif-discovery

# Option B: if already mounted in Drive, just %cd into it
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/circuit-motif-discovery

!bash setup_colab.sh

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Cell 2: Quick Test (optional, ~10 minutes)
!python scripts/run_full_pipeline.py --quick

In [ ]:
# Cell 3: Generate Graphs
!python scripts/01_generate_graphs.py \
  --prompt_file prompts/prompt_corpus.json \
  --output_dir data/graphs \
  --max_prompts_per_family 50

import json
from pathlib import Path
summary = json.loads(Path('data/graphs/summary.json').read_text())
print(summary)

In [ ]:
# Cell 4: Convert to PyG
!python scripts/02_convert_to_pyg.py \
  --input_dir data/graphs \
  --output_path data/circuit_dataset.pt

import torch
dataset = torch.load('data/circuit_dataset.pt', weights_only=False)
print('Num graphs:', len(dataset))
print('Sample graph:', dataset[0])

In [ ]:
# Cell 5: Train GCL
!python scripts/03_train_contrastive.py \
  --dataset_path data/circuit_dataset.pt \
  --output_dir checkpoints/

import torch
ckpt = torch.load('checkpoints/best.pt', map_location='cpu', weights_only=False)
loss_history = ckpt.get('history', [])
print('Best epoch:', ckpt.get('epoch'))
print('Best loss:', ckpt.get('loss'))

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.title('Contrastive Training Loss')
plt.xlabel('Epoch')
plt.ylabel('NT-Xent Loss')
plt.show()

In [ ]:
# Cell 6: Evaluate
!python scripts/04_evaluate.py \
  --dataset_path data/circuit_dataset.pt \
  --checkpoint checkpoints/best.pt \
  --output_dir results/

from IPython.display import Image, display
for p in [
    'results/demo1_umap.png',
    'results/demo2_retrieval.png',
    'results/demo3_cluster_motifs.png',
    'results/demo4_metrics.png',
]:
    display(Image(p))

import json
print(json.dumps(json.load(open('results/metrics_summary.json')), indent=2))

In [ ]:
# Cell 7: Save Results
from pathlib import Path

# Optional: copy to Google Drive after mounting
# !cp -r results /content/drive/MyDrive/circuit-motif-discovery-results

print('Artifacts in results/:')
for p in sorted(Path('results').glob('*')):
    print('-', p)